In [1]:
## Imports and config
from transformers import AutoModelForCausalLM, AutoTokenizer
import json
from pathlib import Path
import random
import re
import copy
import calendar


/home/erfan/miniconda3/envs/hf/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from openai import OpenAI

client = OpenAI(
    base_url="http://127.0.0.1:8080/v1",
    api_key="not-needed",
)

In [3]:

hf_token =json.load(open(Path("./config.json")))["hg_access_token"]
cache_dir =json.load(open(Path("./config.json")))["cache_dir"]
company_names_path=json.load(open(Path("./config.json")))["company_names_path"]
company_names= json.load(open(company_names_path))
timeframes={"daily",'weekly'}
months = {i: calendar.month_name[i] for i in range(1, 13)}


In [4]:
class QueryGenerator:
    def __init__(self, model, tokenizer, system_prompt, max_new_tokens=512, temperature=2.0):
        self.model = model
        self.tokenizer = tokenizer
        self.system_prompt = system_prompt
        self.max_new_tokens = max_new_tokens
        self.temperature = temperature

    def build_messages(self, user_prompt):
        return [
            {"role": "system", "content": self.system_prompt},
            {"role": "user", "content": user_prompt},
        ]

    def build_text(self, user_prompt):
        messages = self.build_messages(user_prompt)

        return self.tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            enable_thinking=False,
            add_generation_prompt=True,
        )

    def generate(self, user_prompt):
        text = self.build_text(user_prompt)

        model_inputs = self.tokenizer(
            [text],
            return_tensors="pt",
        ).to(self.model.device)

        generated_ids = self.model.generate(
            **model_inputs,
            max_new_tokens=self.max_new_tokens,
            temperature=self.temperature,
            top_p= 0.8,
            top_k=20,
            min_p=0 
            
        )

        output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist()

        model_output = self.tokenizer.decode(
            output_ids,
            skip_special_tokens=True,
        ).strip("\n")

        return model_output





class LlamaServerQueryGenerator:
    def __init__(
        self,
        system_prompt,
        base_url="http://127.0.0.1:8080/v1",
        max_tokens=512,
        temperature=0.7,
    ):
        self.client = OpenAI(
            base_url=base_url,
            api_key="not-needed",
        )
        self.system_prompt = system_prompt
        self.max_tokens = max_tokens
        self.temperature = temperature

    def build_messages(self, user_prompt):
        return [
            {
                "role": "system",
                "content": self.system_prompt,
            },
            {
                "role": "user",
                "content": user_prompt,
            },
        ]

    def generate(self, user_prompt):
        response = self.client.chat.completions.create(
            model="local-model",
            messages=self.build_messages(user_prompt),
            max_tokens=self.max_tokens,
            temperature=self.temperature,
            top_p=0.8,
            extra_body={
                "top_k": 20,
                "min_p": 0,
            },
        )

        return response.choices[0].message.content

In [21]:
## Helper functions
PRICE_TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "get_prices",
            "description": (
                "Retrieve historical stock prices for one or more companies "
                "within a specified date range."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "queries": {
                        "type": "array",
                        "items": {
                            "type": "object",
                            "properties": {
                                "symbols": {
                                    "type": "array",
                                    "items": {"type": "string"},
                                },
                                "timeframe": {
                                    "type": "array",
                                    "items": {
                                        "type": "string",
                                        "enum": ["daily", "weekly"],
                                    },
                                },
                                "start_month": {
                                    "type": "string",
                                },
                                "start_year": {
                                    "type": "integer",
                                },
                                "end_month": {
                                    "type": "string",
                                },
                                "end_year": {
                                    "type": "integer",
                                },
                            },
                            "required": [
                                "symbols",
                                "timeframe",
                                "start_month",
                                "start_year",
                                "end_month",
                                "end_year",
                            ],
                        },
                    }
                },
                "required": ["queries"],
            },
        },
    }
]


def sample_spec(company_names, timeframe, months_dict, min_year=2006, max_year=2025):
    company_names_list = list(company_names.keys())
    timeframe_list = list(timeframe)
    
    companies = random.sample(company_names_list, k=random.randint(1, 3))
    timeframes = random.sample(timeframe_list, k=1)

    start_year = random.randint(min_year, max_year)
    end_year = random.randint(start_year, max_year)

    if start_year == end_year:
        # Sample two month keys (1-12) and sort them so month_1 <= month_2
        m1_key, m2_key = sorted(random.sample(list(months_dict.keys()), k=2))
    else:
        # If years are different, any 2 random months are fine
        m1_key, m2_key = random.sample(list(months_dict.keys()), k=2)

    # Convert keys to month names using the dictionary
    selected_months = [months_dict[m1_key], months_dict[m2_key]]

    return companies, timeframes, selected_months, start_year, end_year

def generate_user_pormpt(companies:list,timeframe:list,months,start_year:int,end_year:int):
    company_str= ''
    for comp in companies:
        company_str += comp + ", "
    return f"""Input specification:
    * Companies: {company_str}
    * Time Frames: {timeframe[0]}
    * Start year: {months[0]} of {str(start_year)} 
    * End year: {months[1]} of {str(end_year)}
    Your output :
    """



def build_expected_output(
    companies,
    company_names,
    timeframe,
    months,
    start_year,
    end_year,
):
    arguments = {
        "queries": [
            {
                "symbols": [company_names[company]],
                "timeframe": list(timeframe),
                "start_month": months[0],
                "start_year": start_year,
                "end_month": months[1],
                "end_year": end_year,
            }
            for company in companies
        ]
    }

    return {
        "role": "assistant",
        "content": "",
        "tool_calls": [
            {
                "type": "function",
                "function": {
                    "name": "get_prices",
                    "arguments": arguments,
                },
            }
        ],
    }
    

def parse_model_output(model_output, expected_count=2):
    pattern = r"Q\d+:\s*(.*?)(?=\nQ\d+:|$)"

    queries = re.findall(
        pattern,
        model_output,
        flags=re.S,
    )

    queries = [
        query.strip()
        for query in queries
        if query.strip()
    ]

    if len(queries) != expected_count:
        return None

    return queries

def create_data_points(
    model_output,
    expected_output,
    companies,
    timeframe,
    months,
    start_year,
    end_year,
    company_names,
    tools,
    iteration=None,
):
    queries = parse_model_output(
        model_output,
        expected_count=2,
    )

    metadata = {
        "iteration": iteration,
        "raw_model_output": model_output,
        "companies": list(companies),
        "symbols": [
            company_names[company]
            for company in companies
        ],
        "timeframe": list(timeframe),
        "start_month": months[0],
        "start_year": start_year,
        "end_month": months[1],
        "end_year": end_year,
    }

    if queries is None:
        failed_record = {
            "metadata": metadata,
            "expected_output": copy.deepcopy(expected_output),
        }

        return [], failed_record

    data_points = [
        {
            "messages": [
                {
                    "role": "user",
                    "content": query,
                },
                copy.deepcopy(expected_output),
            ],
            "tools": copy.deepcopy(tools),
            "metadata": copy.deepcopy(metadata),
        }
        for query in queries
    ]

    return data_points, None


def append_jsonl(path, records):
    with open(path, "a", encoding="utf-8") as fp:
        for record in records:
            fp.write(
                json.dumps(record, ensure_ascii=False)
                + "\n"
            )


In [6]:
# model_name = "Qwen/Qwen2.5-7B-Instruct"
# # model_name = "meta-llama/Llama-3.2-3B-Instruct"

# # load the tokenizer and the model
# tokenizer = AutoTokenizer.from_pretrained(model_name, token=hf_token, cache_dir=cache_dir)
# model = AutoModelForCausalLM.from_pretrained(
#     model_name,
#     token=hf_token,
#     cache_dir=cache_dir,
#     torch_dtype="auto",
#     device_map="auto",
    
# )

In [17]:

# prepare the model input
from cmd import PROMPT


SYSTEM_PROMPT = f"""You are an assistant of writing three diffrent querries regarding a dataset. This dataset consist of the weekly and daily prices of stock market companies.

Your resposibilty is to create three queries for function calling to train your assisant. 
Your assistant will use your quesries then call a function. This function will retrieve corresponding prices for the company on specific CSV file.
The user gives you some input specification such as name of comapnies, period of time both in year and month. You must generate three queries based on this informattion  
Each query must contain all information the user inputs. 
Query one has simple grammer and vocabulary. 
Query number two have intermediate vocabualty. 
And query number three has advacned grammer and range of vocabularies.
Put yourself in postion of the user and use natural, common terms and words. Do not use robotic  sentences
Qurries should not be similar from grammer and vocabulary point of view.

You ourput should be like this:
Q1: 
Q2:
Q3:

Rules:
* Preserve every company, preiod of  year and months exactly.
* Do not add or remove any requested information.
* You may vary the sentence structure and the order in which the year range, companies, and period appear.
* Do not use pronouns such as “it,” “the former,” or “the latter.”
* Do not ask for analysis, trends, growth, comparison, explanation, prediction, or recommendation.
"""

# SYSTEM_PROMPT_ADVANCED = f"""You are an assistant of writing twocomplex diffrent queries regarding a dataset. This dataset consist of the weekly and daily prices of stock market companies.

# Your resposibilty is to create two complex queries for function calling to train your assisant. 
# Your assistant will use your quesries then call a function. This function will retrieve corresponding prices for the company on specific CSV file.
# The user gives you some input specification such as name of comapnies, period of time both in year and month. You must generate three queries based on this informattion  
# Each query must contain all information the user inputs and they should be complex from language point of view. Use advanced grammer and vide range of vocabularies.

# Put yourself in postion of the user and use natural, common terms and words. Do not use robotic  sentences
# Qurries should not be similar from grammer and vocabulary point of view.

# You ourput should be like this:
# Q1: 
# Q2:
# Rules:
# * Preserve every company, preiod of  year and months exactly.
# * Do not add or remove any requested information.
# * You may vary the sentence structure and the order in which the year range, companies, and period appear.
# * Do not use pronouns such as “it,” “the former,” or “the latter.”
# * Do not ask for analysis, trends, growth, comparison, explanation, prediction, or recommendation.
# """

SYSTEM_PROMPT_ADVANCED ="""
You generate exactly two linguistically distinct user requests for retrieving historical stock-price data.

## Input information

The input provides:

* one or more company names;
* a frequency: daily or weekly;
* a start month and year;
* an end month and year.

## Task

Write two natural user queries that request the specified stock-price records.

Every query must preserve all input information exactly:

* every company name;
* the requested frequency;
* the start month and year;
* the end month and year.

Do not add companies, dates, operations, filters, or requirements that are absent from the input.

## Diversity requirements

The two queries must differ substantially in syntax, vocabulary, and information order.

### Query 1

* Begin directly with an action verb.
* Use an imperative sentence.
* Mention the company or companies before the date range.
* Do not use “please,” “provide,” “could,” “kindly,” “fetch,” or “specifically.”

### Query 2

* Begin with the date range or the requested frequency.
* Use either a question or a noun-led request.
* Mention the date range before the company or companies.
* Do not reuse any four-word sequence from Query 1.
* Do not use “please,” “provide,” “could,” “kindly,” “fetch,” or “specifically.”

## Language requirements

* Write as a real user requesting data from a financial-data assistant.
* Prefer clear, natural language over formal or decorative wording.
* Vary verbs across outputs, using alternatives such as:

  * retrieve
  * return
  * list
  * show
  * pull
  * obtain
  * extract
  * give me
* Do not repeatedly use phrases such as:

  * “covering the period”
  * “during the timeframe”
  * “specifically focusing on”
  * “from the month of”
  * “inclusive of all weeks”
* Do not use pronouns to replace company names.
* Do not request analysis, comparison, trends, growth, explanation, prediction, summarization, or recommendations.
* Do not state obvious implications such as “include every week” when weekly frequency is already specified.

## Validation before output

Check that:

1. Both queries contain every input field.
2. No new information has been introduced.
3. The two sentence openings are different.
4. The main retrieval verbs are different.
5. The company and date-range order differs.
6. No four-word phrase appears in both queries.

## Output format

Q1: <first query>

Q2: <second query>

Return only the two queries.
"""



In [23]:
all_data_points = []
failed_generations = []

# query_generator= QueryGenerator(model,tokenizer,SYSTEM_PROMPT, max_new_tokens=368,temperature=1)


query_generator_llama_cpp = LlamaServerQueryGenerator(
    system_prompt=SYSTEM_PROMPT_ADVANCED,
    max_tokens=256,
    temperature=1,
)

for i in range(300):
        if (i + 1) % 100 == 0:
            print("iteration:", i+1)
        companies, timeframe, selected_months, start_year, end_year = sample_spec(company_names, timeframes,months)
        user_prompt =  generate_user_pormpt(companies,timeframe,selected_months,start_year,end_year)
        expected_output = build_expected_output(
            companies=companies,
            company_names=company_names,
            timeframe=timeframe,
            months=selected_months,
            start_year=start_year,
            end_year=end_year,
        )        
        model_output = query_generator_llama_cpp.generate(user_prompt)
        data_points, failed_record = create_data_points(
            model_output=model_output,
            expected_output=expected_output,
            companies=companies,
            timeframe=timeframe,
            months=selected_months,
            start_year=start_year,
            end_year=end_year,
            company_names=company_names,
            tools=PRICE_TOOLS,
            iteration=i,
        )

        if data_points:
            all_data_points.extend(data_points)
        else:
            failed_generations.append(failed_record)

        if (i + 1) % 200 == 0:
            append_jsonl(f"dataset_checkpoint_{i + 1}_adv.jsonl", all_data_points)
            append_jsonl(f"failed_checkpoint_{i + 1}._advjsonl", failed_generations)



append_jsonl(f"dataset_checkpoint_adv.jsonl", all_data_points)
append_jsonl(f"failed_checkpoint_adv.jsonl", failed_generations)

iteration: 100
iteration: 200
iteration: 300
